# 07 — FastConformer PC Before/After analysis

This final notebook verifies that PC baseline and final metrics use the same locked manifest. Canonical scores retain the Quranic surface form, while quranic_light scores isolate lexical ASR quality because the PC backbone does not emit diacritics.

In [ ]:
from pathlib import Path
import os

# Each notebook may open in a fresh Colab runtime, so mount Drive before any
# path check rather than relying on a previous notebook's session.
from google.colab import drive
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

# Place the *contents* of this repository in this Google Drive folder, or edit
# this one variable to match the folder you chose.
PROJECT_DIR = DRIVE_ROOT / "quran-fastconformer-colab"
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
import importlib.util
import subprocess
import sys

# A notebook can be opened after a runtime restart. Install project dependencies
# only when a required module is missing, rather than assuming notebook 01 ran.
required_modules = ("datasets", "jiwer", "soundfile", "yaml", "nemo", "pandas", "matplotlib")

missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
needs_numpy_downgrade = False
if not missing_modules:
    import numpy as np
    needs_numpy_downgrade = int(np.__version__.split(".")[0]) >= 2

if missing_modules or needs_numpy_downgrade:
    reason = missing_modules or ["numpy<2 required by the current NeMo audio loader"]
    print("Installing compatible runtime dependencies:", reason)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", "requirements.txt"])
    print("Dependencies updated. Restart the runtime once before launching a NeMo stage.")
else:
    print("Core project dependencies are available.")


In [ ]:
!python -m src.compare --config configs/fastconformer_quran.yaml


In [ ]:
import pandas as pd
from IPython.display import Image, display

COMPARISON_DIR = "artifacts/experiments/fastconformer_pc/results/comparison"
display(pd.read_csv(f"{COMPARISON_DIR}/metrics_comparison.csv"))
display(Image(f"{COMPARISON_DIR}/metrics_comparison.png"))


In [ ]:
print("Performance by reciter")
display(pd.read_csv(f"{COMPARISON_DIR}/metrics_by_reciter.csv"))
print("Performance by duration")
display(pd.read_csv(f"{COMPARISON_DIR}/metrics_by_duration.csv"))
print("Representative predictions")
display(pd.read_csv(f"{COMPARISON_DIR}/prediction_examples.csv").head(12))
